# Обучение детектора жестов «Спортивной мафии» в Colab

Runtime → Change runtime type → **T4 GPU**. Ключ Roboflow вводится через `getpass` и **не сохраняется** в ноутбуке.

In [ ]:
!git clone https://github.com/<ВАШ_НИК>/mafia-gesture-detector.git
%cd mafia-gesture-detector
!pip install -q -r requirements.txt

In [ ]:
import os, getpass
os.environ['ROBOFLOW_API_KEY'] = getpass.getpass('Roboflow API key: ')

## 1. Скачать датасет и проверить утечку train/test по видео

In [ ]:
from mafia_cv.data import download_dataset
data_yaml = download_dataset(out_dir='datasets')
print(data_yaml)
!python -m mafia_cv.leakage {data_yaml.parent}

Если test пересекается с train по видео, сделайте честное разбиение и обучайтесь на нём:

```
!python -m mafia_cv.leakage {data_yaml.parent} --resplit datasets/mafia-cv-grouped
!python -m mafia_cv.train --data datasets/mafia-cv-grouped/data.yaml
```

## 2. Подбор гиперпараметров + обучение + оценка на test

In [ ]:
!python -m mafia_cv.train --tune --data {data_yaml}

## 3. Результаты

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('reports/metrics_test.md', encoding='utf-8').read()))
display(Image('runs/train/mafia-yolov8n/results.png', width=900))
display(Image('runs/val/test/confusion_matrix.png', width=600))

In [ ]:
# скачать веса
from google.colab import files
files.download('models/best.pt')